In [ ]:

import pandas as pd

# Step 1: Read the datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/train.csv'
test_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Step 2: Drop missing values and duplicates
train_df.dropna(inplace=True)
train_df.drop_duplicates(inplace=True)

test_df.dropna(inplace=True)
test_df.drop_duplicates(inplace=True)

print("Train data shape after cleaning:", train_df.shape)
print("Test data shape after cleaning:", test_df.shape)


Train data shape after cleaning: (49415, 19)
Test data shape after cleaning: (12355, 19)


In [ ]:

# Step 3: Feature Construction - Create BMI_Category feature

# Define a function to categorize BMI
def categorize_bmi(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 25:
        return "Normal weight"
    elif 25 <= bmi < 30:
        return "Overweight"
    else:
        return "Obesity"

# Apply the function to create BMI_Category
train_df['BMI'] = train_df['BMI'].astype(float)  # Ensure BMI is in correct format
train_df['BMI_Category'] = train_df['BMI'].apply(categorize_bmi)
train_df['BMI_Category'] = train_df['BMI_Category'].astype('category')

test_df['BMI'] = test_df['BMI'].astype(float)  # Ensure BMI is in correct format
test_df['BMI_Category'] = test_df['BMI'].apply(categorize_bmi)
test_df['BMI_Category'] = test_df['BMI_Category'].astype('category')

print("Train data with BMI_Category:", train_df.head())
print("Test data with BMI_Category:", test_df.head())



##active_line7##
Train data with BMI_Category:   General_Health  ...   BMI_Category
0           Good  ...        Obesity
1           Good  ...  Normal weight
2      Very Good  ...  Normal weight
3      Excellent  ...     Overweight
4      Very Good  ...  Normal weight

[5 rows x 20 columns]
Test data with BMI_Category:   General_Health  ...   BMI_Category
0      Very Good  ...     Overweight
1           Good  ...        Obesity
2      Very Good  ...     Overweight
3      Very Good  ...  Normal weight
4           Fair  ...        Obesity

[5 rows x 20 columns]


In [ ]:


import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Step 4: Model Construction and Training
# Extract features and target variable
features = train_df.drop(columns=['Heart_Disease'])
target = train_df['Heart_Disease']

# Convert categorical columns to numerical
features = pd.get_dummies(features)

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(features, target, test_size=0.2, random_state=42)

# Initialize and train the logistic regression model
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

# Step 5: Prediction and Evaluation
# Convert test data categorical columns to numerical
test_features = pd.get_dummies(test_df.drop(columns=['Heart_Disease']))

# Ensure test features have the same columns as the training features
test_features = test_features.reindex(columns=features.columns, fill_value=0)

# Make predictions on the test set
y_pred_prob = model.predict_proba(test_features)[:, 1]

# Calculate the area under the ROC curve
roc_auc = roc_auc_score(test_df['Heart_Disease'], y_pred_prob)

print(f"Area under ROC curve: {roc_auc:.2f}")



Area under ROC curve: 0.84
